# AH | Preparación dimensional

**Autor:** Alan Arturo Hernández Gutiérrez.

Construye y valida dos hechos y las dimensiones de consumo. Las funciones separan identidad, ventanas, señales, antecedentes y exportación; el cuaderno 02 ejecuta el flujo.

Este cuaderno define las operaciones que utilizan los cuadernos de `notebooks/`. Ejecute sus celdas en orden. El flujo completo se inicia desde 01 o 02.

## Raíz del proyecto

La ubicación se resuelve desde la raíz, `src/` o `notebooks/`, sin guardar rutas personales.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name.lower() in {'src', 'notebooks'}:
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'proyecto.json').is_file():
    raise RuntimeError('Inicie Jupyter desde la raíz del proyecto, src/ o notebooks/.')


## Dependencia territorial

Se carga el cuaderno de enriquecimiento si sus funciones aún no están disponibles.

In [ ]:
if 'preparar_municipios' not in globals():
    cuaderno_dependencia = ROOT / 'src' / 'enriquecimiento.ipynb'
    %run "$cuaderno_dependencia"


## Bibliotecas y parámetros

Se preparan las bibliotecas y constantes utilizadas por las operaciones siguientes.

In [ ]:
from pathlib import Path

from io import BytesIO

import json

import os

import shutil

import tempfile

import uuid

import zipfile

import numpy as np

import pandas as pd

IDS = {
    "TICKET",
    "SKU_CVE",
    "TIENDA_CVE",
    "CUENTA_CVE",
    "CLIENTE_CVE",
    "MUNICIPIO_CVE",
    "ESTADO_CLAVE",
    "MARCA_CVE",
    "CATEGORIA_CVE",
    "SEGMENTO",
}

RULES = [
    (1, "Base observada", "Operaciones tipo 1; importe firmado y todas las categorias"),
    (2, "Solo positivos", "Tipo 1 y monto > 0; sensibilidad, no correccion de devoluciones"),
    (3, "Sin servicios", "Tipo 1, categoria conocida y distinta de SERVICIOS; sensibilidad"),
    (
        4,
        "Positivos sin servicios",
        "Tipo 1, monto > 0 y categoria conocida distinta de SERVICIOS; sensibilidad",
    ),
]

CSV_TABLES = (
    "FactClienteCorte",
    "FactMovimientos",
    "DimCliente",
    "DimProducto",
    "DimTienda",
    "DimReglaAnalisis",
)

DAX_TABLES = ("DimFecha", "DimCorte", "DimTiendaPreferida")


## Serializar resultados del modelo

Representa escalares numéricos, fechas y nulos en los metadatos JSON del modelo.

In [ ]:
def jdefault(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, (pd.Timestamp,)):
        return x.isoformat()
    if pd.isna(x):
        return None
    raise TypeError(type(x).__name__)


## Load sources

Lee la versión identificada por SHA-256 y mantiene los códigos como texto.

In [ ]:
def load_sources(archive, verify=True):
    """Keys stay text; the source files are not extracted or modified."""
    archive = Path(archive)
    config = json.loads((ROOT / "config" / "proyecto.json").read_text(encoding="utf-8"))
    source_sha = sha256_file(archive)
    if (
        verify
        and config.get("datos_internos_sha256")
        and (source_sha != config["datos_internos_sha256"])
    ):
        raise ValueError("El ZIP de origen no coincide con la version revisada")
    frames = {}
    with zipfile.ZipFile(archive) as z:
        for name in [
            "FAC_VENTAS",
            "DIM_CLIENTES",
            "DIM_TIENDA",
            "DIM_PRODUCTO",
            "DIM_CATEGORIA",
            "DIM_MUNICIPIO",
            "DIM_SEGMENTOS",
        ]:
            data = z.read(name + ".csv")
            header = pd.read_csv(BytesIO(data), nrows=0).columns
            dtype = {
                c: "string"
                for c in header
                if c in IDS or c in ["FECHA", "TIPO", "NOM_MUN", "CATEGORIA_NOMBRE"]
            }
            d = pd.read_csv(BytesIO(data), dtype=dtype)
            for c in dtype:
                original = d[c]
                cleaned = original.str.strip()
                d[c] = cleaned.mask(cleaned.eq(""), pd.NA)
            frames[name] = d
    return (frames, config, {"zip_datos_sha256": source_sha})


## Money cents

Comprueba que cada importe pueda representarse con dos decimales antes de convertirlo a centavos.

In [ ]:
def money_cents(s):
    a = pd.to_numeric(s, errors="raise")
    if a.isna().any() or not np.isfinite(a).all():
        raise ValueError("Monto ausente/no finito; no sustituir por cero")
    scaled = a.to_numpy(dtype="float64") * 100.0
    cents = np.rint(scaled)
    if not np.isclose(scaled, cents, rtol=0, atol=1e-06).all():
        raise ValueError(
            "Hay importes con mas de dos decimales; revisar precision antes de redondear"
        )
    return pd.Series(cents.astype("int64"), index=s.index)


## Validar claves de catálogo

Interrumpe la construcción si una clave de dimensión es nula o está repetida; evita multiplicar movimientos durante una unión.

In [ ]:
def unique_lookup(d, key, name):
    if d[key].isna().any() or d[key].duplicated().any():
        raise ValueError(name + ": clave ausente o repetida")


## Preferred

Ordena por fechas distintas, centavos, unidades y clave estable; registra los empates técnicos.

In [ ]:
def preferred(events, column):
    """Full tie-break: distinct visit days, signed cents, units, then stable key."""
    rows = events.loc[events[column].notna()]
    if rows.empty:
        return pd.DataFrame(columns=["ClienteKey", column, "EmpatePreferencia"])
    a = (
        rows.groupby(["ClienteKey", column], dropna=False)
        .agg(
            Visitas=("Fecha", "nunique"),
            Centavos=("MontoCentavos", "sum"),
            Unidades=("Unidades", "sum"),
        )
        .reset_index()
    )
    a = a.sort_values(
        ["ClienteKey", "Visitas", "Centavos", "Unidades", column],
        ascending=[True, False, False, False, True],
        kind="mergesort",
    )
    best = a.drop_duplicates("ClienteKey", keep="first").copy()
    tied = a.merge(
        best[["ClienteKey", "Visitas", "Centavos", "Unidades"]],
        on=["ClienteKey", "Visitas", "Centavos", "Unidades"],
        how="inner",
    )
    tie_n = tied.groupby("ClienteKey").size()
    best["EmpatePreferencia"] = best.ClienteKey.map(tie_n).gt(1).astype("int8")
    return best[["ClienteKey", column, "EmpatePreferencia"]]


## Seleccionar movimientos por regla

Devuelve el filtro de la variante base, positivos, exclusión de servicios o su intersección, manteniendo cada definición separada.

In [ ]:
def rule_mask(tx, key):
    m = tx.OperacionFuente.eq(1)
    if key in (2, 4):
        m = m & tx.MontoCentavos.gt(0)
    if key in (3, 4):
        m = m & tx.CategoriaConocida.eq(1) & tx.CategoriaNombre.ne("SERVICIOS")
    return m


## Actividad por cliente

Consolida la actividad por cliente y clasifica el primer día observado, incluyendo empates de canal.

In [ ]:
def _actividad_por_cliente(e):
    g = e.groupby("ClienteKey")
    s = g.agg(
        FechaPrimeraObservada=("Fecha", "min"),
        FechaUltimaObservada=("Fecha", "max"),
        LineasPeriodo=("LineaFuenteNro", "size"),
        TicketsPeriodo=("TicketKey", "nunique"),
        VisitasPeriodo=("Fecha", "nunique"),
        CentavosPeriodo=("MontoCentavos", "sum"),
        UnidadesPeriodo=("Unidades", "sum"),
    ).reset_index()
    s["FechaPrimeraFisica"] = s.ClienteKey.map(
        e.loc[e.Canal.eq("Fisica")].groupby("ClienteKey").Fecha.min()
    )
    s["FechaPrimeraInternet"] = s.ClienteKey.map(
        e.loc[e.Canal.eq("Internet")].groupby("ClienteKey").Fecha.min()
    )
    first = s.set_index("ClienteKey").FechaPrimeraObservada
    fr = e.loc[e.Fecha.eq(e.ClienteKey.map(first))].copy()
    fr["Fisica"] = fr.Canal.eq("Fisica")
    fr["Internet"] = fr.Canal.eq("Internet")
    fr["Boutique"] = fr.Canal.eq("Boutique")
    fr["Desconocido"] = fr.Canal.eq("Desconocido")
    flags = fr.groupby("ClienteKey")[["Fisica", "Internet", "Boutique", "Desconocido"]].max()
    s = s.merge(flags.reset_index(), on="ClienteKey", validate="one_to_one")
    for flag in ["Fisica", "Internet", "Boutique", "Desconocido"]:
        s[flag] = s[flag].fillna(False).astype(bool)
    s["EstadoOrigen"] = np.select(
        [
            s.Desconocido,
            s.Fisica & s.Internet,
            s.Internet & ~s.Fisica,
            s.Fisica & ~s.Internet & ~s.Boutique,
            s.Boutique & ~s.Fisica & ~s.Internet,
        ],
        [
            "Origen desconocido",
            "Empate fisico-internet",
            "Primero Internet",
            "Elegible fisico",
            "Primero Boutique",
        ],
        default="Otro origen mixto",
    )
    s["EsElegibleFisico"] = s.EstadoOrigen.eq("Elegible fisico").astype("int8")
    s["EsAdoptantePosterior"] = (
        s.EsElegibleFisico.eq(1) & s.FechaPrimeraInternet.gt(s.FechaPrimeraFisica)
    ).astype("int8")
    return s


## Ventanas observacion

Determina seguimiento, cohortes, adopción y recurrencia por horizonte. Una ventana incompleta conserva sus nulos.

In [ ]:
def _ventanas_observacion(e, s, cutoff):
    s["DiasHastaDigital"] = (
        (s.FechaPrimeraInternet - s.FechaPrimeraFisica)
        .dt.days.where(s.EsAdoptantePosterior.eq(1))
        .astype("Int64")
    )
    s["SeguimientoDias"] = (cutoff - s.FechaPrimeraObservada).dt.days.astype("int32")
    s["RecenciaCorteDias"] = (cutoff - s.FechaUltimaObservada).dt.days.astype("int32")
    s["CohorteMes"] = s.FechaPrimeraObservada.dt.to_period("M").astype(str)
    date_map = s.set_index("ClienteKey").FechaPrimeraObservada
    dated = e[["ClienteKey", "Fecha"]].drop_duplicates()
    second = (
        dated.loc[dated.Fecha.gt(dated.ClienteKey.map(date_map))].groupby("ClienteKey").Fecha.min()
    )
    second_dates = pd.Series(second.reindex(s.ClienteKey).to_numpy(), index=s.index)
    days_second = (second_dates - s.FechaPrimeraObservada).dt.days
    for h in (30, 90, 180):
        mature = s.SeguimientoDias.ge(h)
        eligible = s.EsElegibleFisico.eq(1) & mature
        s["VentanaCompleta" + str(h)] = mature.astype("int8")
        s["Adopto" + str(h)] = (
            s.EsAdoptantePosterior.eq(1)
            .mul(s.DiasHastaDigital.le(h).fillna(False))
            .astype("Int64")
            .where(eligible)
        )
        s["Recompra" + str(h)] = days_second.le(h).fillna(False).astype("Int64").where(mature)
    return s


## Senales tempranas

Resume los días 0–30 y separa la adopción de los días 31–90 para mantener el orden temporal.

In [ ]:
def _senales_tempranas(e, s):
    first_physical = s.set_index("ClienteKey").FechaPrimeraFisica
    starts = e.ClienteKey.map(first_physical)
    early = e.loc[
        e.Canal.eq("Fisica") & e.Fecha.ge(starts) & e.Fecha.le(starts + pd.Timedelta(days=30))
    ]
    early_summary = (
        early.groupby("ClienteKey")
        .agg(
            TicketsFisicos30=("TicketKey", "nunique"),
            CategoriasFisicas30=("CategoriaCodigo", "nunique"),
        )
        .reset_index()
    )
    s = s.merge(early_summary, on="ClienteKey", how="left", validate="one_to_one")
    for col in ("TicketsFisicos30", "CategoriasFisicas30"):
        s[col] = s[col].fillna(0).astype("int32")
    s["GrupoTicketsFisicos30"] = pd.cut(
        s.TicketsFisicos30,
        [-1, 0, 1, 3, np.inf],
        labels=["Sin compra fisica", "1 ticket", "2-3 tickets", "4+ tickets"],
    ).astype("string")
    s["GrupoCategoriasFisicas30"] = pd.cut(
        s.CategoriasFisicas30,
        [-1, 0, 1, 2, np.inf],
        labels=["Sin categoria", "1 categoria", "2 categorias", "3+ categorias"],
    ).astype("string")
    candidates = (s.Adopto90.notna() & s.Adopto30.eq(0)).fillna(False)
    s["CandidatoDia30"] = candidates.astype("int8")
    s["Adopto31a90"] = (
        s.DiasHastaDigital.between(31, 90).fillna(False).astype("Int64").where(candidates)
    )
    return s


## Antecedentes cliente

Calcula actividad anterior al día de adopción o hasta el corte. Devuelve los antecedentes y los movimientos admitidos.

In [ ]:
def _antecedentes_cliente(e, s, cutoff):
    ref = s.FechaPrimeraInternet.where(s.EsAdoptantePosterior.eq(1), cutoff)
    s["FechaReferenciaAntecedentes"] = ref
    s["AntiguedadReferenciaDias"] = (ref - s.FechaPrimeraObservada).dt.days.astype("Int64")
    refs = pd.Series(ref.values, index=s.ClienteKey)
    adopters = s.set_index("ClienteKey").EsAdoptantePosterior
    ev_adopt = e.ClienteKey.map(adopters).eq(1)
    before = e.loc[
        ev_adopt & e.Fecha.lt(e.ClienteKey.map(refs)) | ~ev_adopt & e.Fecha.le(cutoff)
    ].copy()
    ag = (
        before.groupby("ClienteKey")
        .agg(
            VisitasAntecedentes=("Fecha", "nunique"),
            TicketsAntecedentes=("TicketKey", "nunique"),
            LineasAntecedentes=("LineaFuenteNro", "size"),
            CentavosAntecedentes=("MontoCentavos", "sum"),
            UnidadesAntecedentes=("Unidades", "sum"),
            FechaUltimaAntecedente=("Fecha", "max"),
        )
        .reset_index()
    )
    s = s.merge(ag, on="ClienteKey", how="left", validate="one_to_one")
    for c in [
        "VisitasAntecedentes",
        "TicketsAntecedentes",
        "LineasAntecedentes",
        "CentavosAntecedentes",
        "UnidadesAntecedentes",
    ]:
        s[c] = s[c].fillna(0)
    s["InactividadReferenciaDias"] = (
        s.FechaReferenciaAntecedentes - s.FechaUltimaAntecedente
    ).dt.days.astype("Int64")
    return (s, before)


## Importes y preferencias

Conserva centavos y signos y determina preferencias mediante desempates estables.

In [ ]:
def _importes_y_preferencias(e, s, before, cutoff, rulekey):
    positives = e.loc[e.MontoCentavos.gt(0)].groupby("ClienteKey").MontoCentavos.sum()
    negatives = e.loc[e.MontoCentavos.lt(0)].groupby("ClienteKey").MontoCentavos.sum()
    s["CentavosPositivosPeriodo"] = s.ClienteKey.map(positives).fillna(0).astype("int64")
    s["CentavosNegativosPeriodo"] = s.ClienteKey.map(negatives).fillna(0).astype("int64")
    store = preferred(before, "TiendaKey").rename(
        columns={"TiendaKey": "TiendaPreferidaKey", "EmpatePreferencia": "EmpateTiendaPreferida"}
    )
    cat = preferred(before, "CategoriaCodigo").rename(
        columns={
            "CategoriaCodigo": "CategoriaPreferidaCodigo",
            "EmpatePreferencia": "EmpateCategoriaPreferida",
        }
    )
    s = s.merge(store, on="ClienteKey", how="left", validate="one_to_one").merge(
        cat, on="ClienteKey", how="left", validate="one_to_one"
    )
    s["TiendaPreferidaKey"] = s.TiendaPreferidaKey.fillna(0).astype("int32")
    s["CategoriaPreferidaCodigo"] = s.CategoriaPreferidaCodigo.fillna("Desconocida")
    for c in ["EmpateTiendaPreferida", "EmpateCategoriaPreferida"]:
        s[c] = s[c].fillna(0).astype("int8")
    s["CorteKey"] = int(cutoff.strftime("%Y%m%d"))
    s["ReglaKey"] = rulekey
    for c in list(s.columns):
        if c.startswith("Centavos"):
            s[c] = s[c].astype("int64")
            s[c.replace("Centavos", "Importe", 1)] = s[c] / 100.0
    s["ImportePorTicketPeriodo"] = s.ImportePeriodo / s.TicketsPeriodo.where(s.TicketsPeriodo.gt(0))
    s["GrupoRecencia"] = pd.cut(
        s.RecenciaCorteDias,
        [-1, 30, 90, np.inf],
        labels=["0-30 dias", "31-90 dias", "Mas de 90 dias"],
    ).astype("string")
    s["GrupoFrecuencia"] = pd.cut(
        s.TicketsPeriodo, [0, 1, 5, np.inf], labels=["1 ticket", "2-5 tickets", "6 o mas tickets"]
    ).astype("string")
    return s


## Snapshot

Coordina las transformaciones de un cliente–corte–regla; cada etapa tiene una función específica.

In [ ]:
def snapshot(events, cutoff, rulekey):
    e = events.loc[events.Fecha.le(cutoff)].copy()
    if e.empty:
        raise ValueError("La regla/corte deja el universo vacio")
    s = _actividad_por_cliente(e)
    s = _ventanas_observacion(e, s, cutoff)
    s = _senales_tempranas(e, s)
    s, before = _antecedentes_cliente(e, s, cutoff)
    s = _importes_y_preferencias(e, s, before, cutoff, rulekey)
    return (
        s.drop(columns=["Fisica", "Internet", "Boutique", "Desconocido"])
        .sort_values("ClienteKey")
        .reset_index(drop=True)
    )


## Preparar movimientos

Valida claves, granularidad, fechas, identidad de ticket y precisión monetaria antes de unir las dimensiones.

In [ ]:
def _preparar_movimientos(frames, config):
    f = frames["FAC_VENTAS"].copy()
    for table, key in [
        ("DIM_CLIENTES", "CUENTA_CVE"),
        ("DIM_TIENDA", "TIENDA_CVE"),
        ("DIM_PRODUCTO", "SKU_CVE"),
        ("DIM_CATEGORIA", "CATEGORIA_CVE"),
        ("DIM_MUNICIPIO", "MUNICIPIO_CVE"),
        ("DIM_SEGMENTOS", "CLIENTE_CVE"),
    ]:
        unique_lookup(frames[table], key, table)
    if f[["TICKET", "SKU_CVE", "TIENDA_CVE", "CUENTA_CVE", "FECHA"]].isna().any().any():
        raise ValueError("Hechos: clave esencial ausente")
    if f.duplicated(["TICKET", "SKU_CVE"]).any():
        raise ValueError("Hechos: ticket-SKU repetido; revisar grano, no eliminar silenciosamente")
    if not f.FECHA.str.fullmatch("\\d{4}-\\d{2}-\\d{2}").all():
        raise ValueError("Formato de fecha no ISO; revisar fuente")
    f["Fecha"] = pd.to_datetime(f.FECHA, format="%Y-%m-%d", errors="raise")
    f["MontoCentavos"] = money_cents(f.MONTO)
    f["LineaFuenteNro"] = np.arange(1, len(f) + 1, dtype="int32")
    f["TicketKey"] = pd.factorize(f.TICKET, sort=True)[0].astype("int32") + 1
    if (
        f.groupby("TicketKey").Fecha.nunique().gt(1)
        | f.groupby("TicketKey").TIENDA_CVE.nunique().gt(1)
        | f.groupby("TicketKey").CUENTA_CVE.nunique().gt(1)
    ).any():
        raise ValueError(
            "Un ticket corresponde a varias fechas/tiendas/cuentas; revisar su identidad"
        )
    cutoff = pd.Timestamp(config["fecha_corte"]) if config.get("fecha_corte") else f.Fecha.max()
    if cutoff < f.Fecha.min() or cutoff > f.Fecha.max():
        raise ValueError("Fecha de corte fuera de la observacion fuente")
    return (f, cutoff)


## Dimension clientes

Consolida cuentas por cliente, comprueba atributos y agrega cobertura y contexto municipal.

In [ ]:
def _dimension_clientes(frames, f, municipios):
    cli = frames["DIM_CLIENTES"]
    used_accounts = f.CUENTA_CVE.unique()
    if not pd.Series(used_accounts).isin(cli.CUENTA_CVE).all():
        raise ValueError("Cuenta de movimiento sin cliente; no asignar persona por suposicion")
    active_ids = cli.loc[cli.CUENTA_CVE.isin(used_accounts), "CLIENTE_CVE"].unique()
    active = cli.loc[cli.CLIENTE_CVE.isin(active_ids)].copy()
    if (
        active.groupby("CLIENTE_CVE").EDAD.nunique().gt(1).any()
        or active.groupby("CLIENTE_CVE").MUNICIPIO_CVE.nunique().gt(1).any()
    ):
        raise ValueError("Atributos de cliente contradictorios entre cuentas; resolver vigencia")
    clients = (
        active.groupby("CLIENTE_CVE")
        .agg(
            EdadObservada=("EDAD", "first"),
            MunicipioDomicilioCve=("MUNICIPIO_CVE", "first"),
            CuentasEnMaestro=("CUENTA_CVE", "nunique"),
        )
        .reset_index()
        .sort_values("CLIENTE_CVE")
    )
    clients.insert(0, "ClienteKey", np.arange(1, len(clients) + 1, dtype="int32"))
    customer_map = clients[["ClienteKey", "CLIENTE_CVE"]].copy()
    clients["ClienteReferencia"] = clients.ClienteKey.map(lambda x: "C{:06d}".format(x))
    clients = clients.merge(
        frames["DIM_SEGMENTOS"].rename(columns={"SEGMENTO": "SegmentoOrigenCodigo"}),
        on="CLIENTE_CVE",
        how="left",
        validate="one_to_one",
    )
    clients["EdadDisponible"] = clients.EdadObservada.notna().astype("int8")
    clients["EdadMayor100Revisar"] = clients.EdadObservada.gt(100).fillna(False).astype("int8")
    clients["GrupoEdad"] = (
        pd.cut(
            clients.EdadObservada,
            [-np.inf, 17, 29, 44, 59, 74, np.inf],
            labels=["Menos de 18", "18-29", "30-44", "45-59", "60-74", "75 o mas"],
        )
        .astype("string")
        .fillna("Sin dato")
    )
    clients["SegmentoDisponible"] = clients.SegmentoOrigenCodigo.notna().astype("int8")
    clients["SegmentoOrigenCodigo"] = clients.SegmentoOrigenCodigo.fillna("Sin dato")
    clients = clients.merge(
        municipios.rename(columns={"MunicipioCve": "MunicipioDomicilioCve"}),
        on="MunicipioDomicilioCve",
        how="left",
        validate="many_to_one",
    )
    clients["DomicilioMunicipioValido"] = clients.MunicipioNombre.notna().astype("int8")
    clients["CensoMunicipalDisponible"] = clients.CensoMunicipalDisponible.fillna(0).astype("int8")
    clients = clients.drop(columns="CLIENTE_CVE")
    return (clients, customer_map, cli)


## Dimension productos

Asigna una clave de producto y conserva las categorías y marcas codificadas, incluidos miembros desconocidos.

In [ ]:
def _dimension_productos(frames, f):
    products = (
        f[["SKU_CVE"]]
        .drop_duplicates()
        .sort_values("SKU_CVE")
        .merge(frames["DIM_PRODUCTO"], on="SKU_CVE", how="left", validate="one_to_one")
    )
    products.insert(0, "ProductoKey", np.arange(1, len(products) + 1, dtype="int32"))
    products["ProductoEnCatalogo"] = products.CATEGORIA_CVE.notna().astype("int8")
    products = products.merge(
        frames["DIM_CATEGORIA"], on="CATEGORIA_CVE", how="left", validate="many_to_one"
    )
    products = products.rename(
        columns={
            "SKU_CVE": "ProductoCodigoFuente",
            "MARCA_CVE": "MarcaCodigo",
            "CATEGORIA_CVE": "CategoriaCodigo",
            "CATEGORIA_NOMBRE": "CategoriaNombre",
        }
    )
    products["MarcaCategoriaCodigo"] = (
        products.CategoriaCodigo + ":" + products.MarcaCodigo
    ).fillna("Desconocida")
    products["CategoriaCodigo"] = products.CategoriaCodigo.fillna("Desconocida")
    products["CategoriaNombre"] = products.CategoriaNombre.fillna("Desconocida")
    products["MarcaCodigo"] = products.MarcaCodigo.fillna("Desconocida")
    return products


## Dimension tiendas

Asigna una clave de tienda y conserva canal, ubicación comercial y banderas de cobertura.

In [ ]:
def _dimension_tiendas(frames, f, municipios):
    stores = (
        f[["TIENDA_CVE"]]
        .drop_duplicates()
        .sort_values("TIENDA_CVE")
        .merge(frames["DIM_TIENDA"], on="TIENDA_CVE", how="left", validate="one_to_one")
    )
    stores.insert(0, "TiendaKey", np.arange(1, len(stores) + 1, dtype="int32"))
    stores["TiendaEnCatalogo"] = stores.TIPO.notna().astype("int8")
    stores = stores.rename(
        columns={
            "TIENDA_CVE": "TiendaCodigoFuente",
            "TIPO": "Canal",
            "MUNICIPIO_CVE": "MunicipioTiendaCve",
        }
    )
    stores["Canal"] = stores.Canal.fillna("Desconocido")
    stores = stores.merge(
        municipios.rename(columns={"MunicipioCve": "MunicipioTiendaCve"}),
        on="MunicipioTiendaCve",
        how="left",
        validate="many_to_one",
    )
    stores["MunicipioTiendaValido"] = stores.MunicipioNombre.notna().astype("int8")
    stores["CensoMunicipalDisponible"] = stores.CensoMunicipalDisponible.fillna(0).astype("int8")
    return stores


## Hecho enriquecido

Une los movimientos con claves sustitutas y comprueba que filas e importes permanezcan iguales.

In [ ]:
def _hecho_enriquecido(f, cli, customer_map, products, stores):
    tx = f.merge(
        cli[["CUENTA_CVE", "CLIENTE_CVE"]], on="CUENTA_CVE", how="left", validate="many_to_one"
    ).merge(customer_map, on="CLIENTE_CVE", how="left", validate="many_to_one")
    tx = tx.merge(
        products[
            [
                "ProductoKey",
                "ProductoCodigoFuente",
                "CategoriaCodigo",
                "CategoriaNombre",
                "ProductoEnCatalogo",
            ]
        ],
        left_on="SKU_CVE",
        right_on="ProductoCodigoFuente",
        how="left",
        validate="many_to_one",
    )
    tx = tx.merge(
        stores[["TiendaKey", "TiendaCodigoFuente", "Canal", "TiendaEnCatalogo"]],
        left_on="TIENDA_CVE",
        right_on="TiendaCodigoFuente",
        how="left",
        validate="many_to_one",
    )
    tx = tx.rename(
        columns={
            "ITEMS": "Unidades",
            "TIPO_OPERACION": "OperacionFuente",
            "ProductoEnCatalogo": "CategoriaConocida",
        }
    )
    tx["FechaKey"] = tx.Fecha.dt.strftime("%Y%m%d").astype("int32")
    tx["ImporteFirmado"] = tx.MontoCentavos / 100.0
    tx["EsMontoNegativo"] = tx.MontoCentavos.lt(0).astype("int8")
    tx["EsMontoCero"] = tx.MontoCentavos.eq(0).astype("int8")
    tx["EsServicio"] = tx.CategoriaNombre.eq("SERVICIOS").astype("int8")
    tx["EsProductoDesconocido"] = tx.CategoriaConocida.eq(0).astype("int8")
    tx["EsCanalDesconocido"] = tx.Canal.eq("Desconocido").astype("int8")
    for c in ["ClienteKey", "ProductoKey", "TiendaKey"]:
        if tx[c].isna().any():
            raise ValueError("Clave sustituta sin asignar: " + c)
        tx[c] = tx[c].astype("int32")
    if len(tx) != len(f) or int(tx.MontoCentavos.sum()) != int(f.MontoCentavos.sum()):
        raise AssertionError("Las uniones cambiaron filas o importes")
    return tx


## Snapshots enriquecidos

Aplica las cuatro reglas y distingue domicilio observado de ubicación comercial por tienda preferida.

In [ ]:
def _snapshots_enriquecidos(tx, cutoff, clients, stores, frames):
    snapshots = []
    for rk, name, definition in RULES:
        print("Construyendo regla {}: {}".format(rk, name), flush=True)
        snapshots.append(snapshot(tx.loc[rule_mask(tx, rk)], cutoff, rk))
    snap = pd.concat(snapshots, ignore_index=True)
    home = clients.set_index("ClienteKey").MunicipioDomicilioCve
    home_valid = clients.set_index("ClienteKey").DomicilioMunicipioValido
    store_geo = stores.set_index("TiendaKey").MunicipioTiendaCve
    store_valid = stores.set_index("TiendaKey").MunicipioTiendaValido
    domicile = snap.ClienteKey.map(home).where(snap.ClienteKey.map(home_valid).eq(1))
    proxy = snap.TiendaPreferidaKey.map(store_geo).where(
        snap.TiendaPreferidaKey.map(store_valid).eq(1)
    )
    snap["MunicipioComercialCve"] = domicile.fillna(proxy)
    snap["OrigenGeografiaComercial"] = np.select(
        [domicile.notna(), proxy.notna()],
        ["Domicilio observado", "Proxy tienda preferida"],
        default="Sin dato",
    )
    snap["CategoriaPreferidaNombre"] = snap.CategoriaPreferidaCodigo.map(
        frames["DIM_CATEGORIA"].set_index("CATEGORIA_CVE").CATEGORIA_NOMBRE
    ).fillna("Desconocida")
    return snap


## Calendario referencia

Construye el calendario en memoria para comprobar las claves de fecha del modelo.

In [ ]:
def _calendario_referencia(snap, f):
    dates = pd.DataFrame(
        {
            "Fecha": pd.date_range(
                snap.FechaPrimeraObservada.min(), snap.FechaUltimaObservada.max(), freq="D"
            )
        }
    )
    dates.insert(0, "FechaKey", dates.Fecha.dt.strftime("%Y%m%d").astype("int32"))
    dates["Anio"] = dates.Fecha.dt.year
    dates["MesNumero"] = dates.Fecha.dt.month
    names = [
        "Enero",
        "Febrero",
        "Marzo",
        "Abril",
        "Mayo",
        "Junio",
        "Julio",
        "Agosto",
        "Septiembre",
        "Octubre",
        "Noviembre",
        "Diciembre",
    ]
    dates["MesNombre"] = dates.MesNumero.map(dict(enumerate(names, 1)))
    dates["AnioMes"] = dates.Fecha.dt.strftime("%Y-%m")
    dates["Trimestre"] = dates.Fecha.dt.quarter
    dates["DiaSemanaNumero"] = dates.Fecha.dt.dayofweek + 1
    dates["EsFinSemana"] = dates.DiaSemanaNumero.ge(6).astype("int8")
    dates["DiaConRegistrosFuente"] = dates.Fecha.isin(f.Fecha).astype("int8")
    return dates


## Build model

Coordina dimensiones, hechos, snapshots y metadatos. Las tres dimensiones DAX se usan como referencia de validación.

In [ ]:
def build_model(frames, config, iter_path=None):
    f, cutoff = _preparar_movimientos(frames, config)
    municipios, enrichment = preparar_municipios(
        frames["DIM_MUNICIPIO"], ROOT / "catalogos" / "estados_inegi.csv", iter_path
    )
    clients, customer_map, cli = _dimension_clientes(frames, f, municipios)
    products = _dimension_productos(frames, f)
    stores = _dimension_tiendas(frames, f, municipios)
    tx = _hecho_enriquecido(f, cli, customer_map, products, stores)
    snap = _snapshots_enriquecidos(tx, cutoff, clients, stores, frames)
    pref = stores.rename(columns={"TiendaKey": "TiendaPreferidaKey"}).copy()
    dates = _calendario_referencia(snap, f)
    cols_fact = [
        "LineaFuenteNro",
        "TicketKey",
        "FechaKey",
        "ClienteKey",
        "ProductoKey",
        "TiendaKey",
        "OperacionFuente",
        "MontoCentavos",
        "ImporteFirmado",
        "Unidades",
        "EsMontoNegativo",
        "EsMontoCero",
        "EsServicio",
        "EsProductoDesconocido",
        "EsCanalDesconocido",
    ]
    tables = {
        "DimFecha": dates,
        "DimCliente": clients.sort_values("ClienteKey").reset_index(drop=True),
        "DimProducto": products.sort_values("ProductoKey").reset_index(drop=True),
        "DimTienda": stores.sort_values("TiendaKey").reset_index(drop=True),
        "DimTiendaPreferida": pref.sort_values("TiendaPreferidaKey").reset_index(drop=True),
        "DimReglaAnalisis": pd.DataFrame(RULES, columns=["ReglaKey", "ReglaNombre", "Definicion"]),
        "DimCorte": pd.DataFrame(
            [
                {
                    "CorteKey": int(cutoff.strftime("%Y%m%d")),
                    "FechaCorte": cutoff,
                    "CorteNombre": cutoff.strftime("%Y-%m-%d"),
                }
            ]
        ),
        "FactMovimientos": tx[cols_fact].sort_values("LineaFuenteNro").reset_index(drop=True),
        "FactClienteCorte": snap.sort_values(["CorteKey", "ReglaKey", "ClienteKey"]).reset_index(
            drop=True
        ),
    }
    lineage = customer_map.rename(columns={"CLIENTE_CVE": "ClienteCodigoFuente"})
    metadata = {
        "fecha_corte": cutoff.strftime("%Y-%m-%d"),
        "enriquecimiento": enrichment,
        "filas_fuentes": {k: len(v) for k, v in frames.items()},
        "referencias_fuente": {
            "moneda": config["moneda"],
            "interpretacion_importes": config["interpretacion_importes"],
        },
    }
    return (tables, municipios, lineage, tx, metadata)


## Validar claves, grano y totales

Registra unicidad de dimensiones, correspondencia de claves, granularidad de hechos y preservación de filas e importes.

In [ ]:
def _validar_integridad_tablas(tables, tx, metadata, check):
    f = tables["FactMovimientos"]
    s = tables["FactClienteCorte"]
    for table, key in [
        ("DimFecha", "FechaKey"),
        ("DimCliente", "ClienteKey"),
        ("DimProducto", "ProductoKey"),
        ("DimTienda", "TiendaKey"),
        ("DimTiendaPreferida", "TiendaPreferidaKey"),
        ("DimReglaAnalisis", "ReglaKey"),
        ("DimCorte", "CorteKey"),
    ]:
        check(
            "Unicidad " + table,
            tables[table][key].notna().all() and (not tables[table][key].duplicated().any()),
            len(tables[table]),
        )
    for fact, fk, dim, pk in [
        ("FactMovimientos", "FechaKey", "DimFecha", "FechaKey"),
        ("FactMovimientos", "ClienteKey", "DimCliente", "ClienteKey"),
        ("FactMovimientos", "ProductoKey", "DimProducto", "ProductoKey"),
        ("FactMovimientos", "TiendaKey", "DimTienda", "TiendaKey"),
        ("FactClienteCorte", "ClienteKey", "DimCliente", "ClienteKey"),
        ("FactClienteCorte", "TiendaPreferidaKey", "DimTiendaPreferida", "TiendaPreferidaKey"),
        ("FactClienteCorte", "ReglaKey", "DimReglaAnalisis", "ReglaKey"),
        ("FactClienteCorte", "CorteKey", "DimCorte", "CorteKey"),
    ]:
        check(
            "FK " + fact + "." + fk,
            tables[fact][fk].isin(tables[dim][pk]).all(),
            "Sin claves huerfanas",
        )
    check(
        "Grano FactMovimientos",
        not f.LineaFuenteNro.duplicated().any(),
        "Una linea de fuente por fila",
    )
    check(
        "Grano FactClienteCorte",
        not s.duplicated(["ClienteKey", "CorteKey", "ReglaKey"]).any(),
        "Cliente/corte/regla",
    )
    check("Filas preservadas", len(f) == metadata["filas_fuentes"]["FAC_VENTAS"], len(f))
    check(
        "Importes exactos preservados",
        int(f.MontoCentavos.sum()) == int(tx.MontoCentavos.sum()),
        int(f.MontoCentavos.sum()),
    )
    check(
        "Canales desconocidos conservados",
        int(f.EsCanalDesconocido.sum()) == int(tx.Canal.eq("Desconocido").sum()),
        int(f.EsCanalDesconocido.sum()),
    )
    check(
        "Negativos conservados",
        int(f.EsMontoNegativo.sum()) == int(tx.MontoCentavos.lt(0).sum()),
        int(f.EsMontoNegativo.sum()),
    )
    check(
        "No falsos negativos temporales",
        s.DiasHastaDigital.dropna().gt(0).all()
        and s.InactividadReferenciaDias.dropna().ge(0).all(),
        "Dias validos",
    )


## Validar nulos por seguimiento

Comprueba que adopción y recurrencia tengan dato únicamente para la población y el seguimiento que corresponde a cada horizonte.

In [ ]:
def _validar_horizontes(tables, check):
    s = tables["FactClienteCorte"]
    for h in (30, 90, 180):
        flag = s["Adopto" + str(h)]
        ready = s.EsElegibleFisico.eq(1) & s["VentanaCompleta" + str(h)].eq(1)
        check(
            "Nulos correctos adopcion " + str(h),
            flag.notna().eq(ready).all(),
            "No elegible/inmaduro = nulo",
        )
        check(
            "Recompra solo madura " + str(h),
            s["Recompra" + str(h)].notna().eq(s["VentanaCompleta" + str(h)].eq(1)).all(),
            "Ventana completa",
        )


## Conciliar reglas y antecedentes

Contrasta clientes, centavos y adopción por variante y recalcula antecedentes sobre una selección representativa de eventos.

In [ ]:
def _validar_reglas(tables, tx, metadata, check):
    s = tables["FactClienteCorte"]
    cutoff = pd.Timestamp(metadata["fecha_corte"])
    for rk, _, _ in RULES:
        e = tx.loc[rule_mask(tx, rk) & tx.Fecha.le(cutoff)]
        x = s.loc[s.ReglaKey.eq(rk)]
        check("Clientes regla " + str(rk), len(x) == e.ClienteKey.nunique(), len(x))
        check(
            "Importe snapshot regla " + str(rk),
            int(x.CentavosPeriodo.sum()) == int(e.MontoCentavos.sum()),
            int(x.CentavosPeriodo.sum()),
        )
        day30 = x.CandidatoDia30.eq(1)
        check(
            "Ventana dia 30 regla " + str(rk),
            x.Adopto31a90.notna().eq(day30).all() and x.loc[day30, "TicketsFisicos30"].ge(1).all(),
            "Candidatos con actividad fisica inicial y resultado definido",
        )
        check(
            "Adopcion 31-90 regla " + str(rk),
            int(x.Adopto31a90.sum()) == int(x.Adopto90.sum() - x.Adopto30.sum()),
            "Horizontes 90 menos 30",
        )
        candidates = pd.concat(
            [
                x.head(3),
                x.loc[x.EsAdoptantePosterior.eq(1)].head(3),
                x.loc[x.EsElegibleFisico.eq(0)].head(2),
            ]
        ).drop_duplicates("ClienteKey")
        for _, r in candidates.iterrows():
            q = e.loc[e.ClienteKey.eq(r.ClienteKey)]
            pre = q.loc[q.Fecha.lt(r.FechaPrimeraInternet)] if r.EsAdoptantePosterior == 1 else q
            check(
                "Visitas antecedentes r{} c{}".format(rk, int(r.ClienteKey)),
                int(r.VisitasAntecedentes) == pre.Fecha.nunique(),
                "Conteo independiente de fechas",
            )
            check(
                "Inactividad antecedentes r{} c{}".format(rk, int(r.ClienteKey)),
                (
                    pd.isna(r.InactividadReferenciaDias)
                    if pre.empty
                    else int(r.InactividadReferenciaDias)
                    == int((r.FechaReferenciaAntecedentes - pre.Fecha.max()).days)
                ),
                "Referencia menos ultima fecha previa",
            )


## Validate model

Comprueba granularidad, claves, cobertura, totales, ventanas y antecedentes antes de exportar.

In [ ]:
def validate_model(tables, tx, metadata):
    results = []

    def check(name, ok, detail):
        results.append(
            {"Prueba": name, "Resultado": "OK" if bool(ok) else "FALLO", "Detalle": str(detail)}
        )

    _validar_integridad_tablas(tables, tx, metadata, check)
    _validar_horizontes(tables, check)
    _validar_reglas(tables, tx, metadata, check)
    qa = pd.DataFrame(results)
    if qa.Resultado.eq("FALLO").any():
        raise AssertionError(qa.loc[qa.Resultado.eq("FALLO")].to_string(index=False))
    return qa


## Summarize

Produce los agregados de reglas, cohortes y territorio que sirven para conciliar el reporte.

In [ ]:
def summarize(tables):
    s = tables["FactClienteCorte"]
    customers = tables["DimCliente"]
    rows = []
    for rk, name, _ in RULES:
        x = s.loc[s.ReglaKey.eq(rk)]
        den = int(x.Adopto90.notna().sum())
        num = int(x.Adopto90.fillna(0).sum())
        ordered = x.CentavosPositivosPeriodo.sort_values(ascending=False)
        top = max(1, int(np.ceil(len(x) * 0.1)))
        pos = int(ordered.sum())
        rows.append(
            {
                "ReglaKey": rk,
                "ReglaNombre": name,
                "ClientesObservados": len(x),
                "ClientesElegiblesFisicos": int(x.EsElegibleFisico.sum()),
                "Elegibles90Dias": den,
                "Adoptantes90Dias": num,
                "TasaAdopcion90": num / den if den else np.nan,
                "ImporteRegistrado": int(x.CentavosPeriodo.sum()) / 100.0,
                "ParticipacionTop10PctImportePositivo": (
                    int(ordered.head(top).sum()) / pos if pos else np.nan
                ),
                "ClientesTop10Pct": top,
                "Recurrencia90": float(x.Recompra90.mean()),
            }
        )
    cohort = (
        s.loc[s.EsElegibleFisico.eq(1)]
        .groupby(["ReglaKey", "CohorteMes"])
        .agg(
            ClientesElegibles=("ClienteKey", "size"),
            Elegibles90=("Adopto90", "count"),
            Adoptantes90=("Adopto90", "sum"),
        )
        .reset_index()
    )
    cohort["TasaAdopcion90"] = cohort.Adoptantes90 / cohort.Elegibles90.where(
        cohort.Elegibles90.gt(0)
    )
    geo = s.merge(
        customers[
            [
                "ClienteKey",
                "MunicipioDomicilioCve",
                "MunicipioNombre",
                "NombreEstado",
                "DomicilioMunicipioValido",
            ]
        ],
        on="ClienteKey",
        how="left",
        validate="many_to_one",
    )
    territory = (
        geo.loc[geo.DomicilioMunicipioValido.eq(1)]
        .groupby(["ReglaKey", "MunicipioDomicilioCve", "MunicipioNombre", "NombreEstado"])
        .agg(
            ClientesObservados=("ClienteKey", "size"),
            Elegibles90=("Adopto90", "count"),
            Adoptantes90=("Adopto90", "sum"),
            CentavosPositivos=("CentavosPositivosPeriodo", "sum"),
        )
        .reset_index()
    )
    territory["TasaAdopcion90"] = territory.Adoptantes90 / territory.Elegibles90.where(
        territory.Elegibles90.gt(0)
    )
    territory["ImportePositivo"] = territory.CentavosPositivos / 100.0
    return {
        "SensibilidadReglas": pd.DataFrame(rows),
        "Cohortes90": cohort,
        "OportunidadesTerritoriales": territory,
    }


## Describir tipos de exportación

Asigna tipos de fecha, entero, número, lógico o texto para el esquema de auditoría.

In [ ]:
def schema_type(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "date"
    if pd.api.types.is_integer_dtype(series):
        return "int64"
    if pd.api.types.is_float_dtype(series):
        return "number"
    if pd.api.types.is_bool_dtype(series):
        return "logical"
    return "text"


## Escribir el contrato CSV

Exporta encabezados y registros en UTF-8 con BOM, fechas ISO, nulos vacíos y saltos de línea uniformes.

In [ ]:
def csv_export(df, path):
    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
        date_format="%Y-%m-%d",
        float_format="%.15g",
        na_rep="",
        lineterminator="\n",
    )


## Export release

Construye una salida temporal, relee los CSV y promueve la versión validada.

In [ ]:
def export_release(tables, municipios, lineage, analyses, qa, metadata, output_root):
    output_root = Path(output_root).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix="_build_", dir=str(output_root))).resolve()
    if stage.parent != output_root:
        raise ValueError("Directorio temporal fuera del destino autorizado")
    for folder in ["modelo", "analisis", "auditoria", "catalogos", "trazabilidad_privada"]:
        (stage / folder).mkdir()
    schemas = {}
    for name in CSV_TABLES:
        d = tables[name]
        csv_export(d, stage / "modelo" / (name + ".csv"))
        schemas[name] = {
            "rows": len(d),
            "columns": [{"name": c, "type": schema_type(d[c])} for c in d],
        }
    csv_export(municipios, stage / "catalogos" / "MunicipiosEnriquecidos.csv")
    csv_export(lineage, stage / "trazabilidad_privada" / "MapaClienteFuente.csv")
    for name, d in analyses.items():
        csv_export(d, stage / "analisis" / (name + ".csv"))
    csv_export(qa, stage / "auditoria" / "Validaciones.csv")
    (stage / "auditoria" / "EsquemaModelo.json").write_text(
        json.dumps(schemas, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    metadata["validaciones"] = {
        "ejecutadas": len(qa),
        "fallidas": int(qa.Resultado.eq("FALLO").sum()),
    }
    metadata["tablas_csv"] = list(CSV_TABLES)
    metadata["dimensiones_dax"] = list(DAX_TABLES)
    metadata["limites"] = [
        "Power BI Desktop requiere validacion de consultas y medidas",
        "Muestra transaccional con codigos de origen e indicadores municipales agregados",
        "Montos negativos/servicios sin semantica empresarial acreditada",
        "Contexto municipal Censo solo si se aporta copia local; campos ausentes no se imputan",
    ]
    metadata["archivos"] = {
        p.relative_to(stage).as_posix(): {"sha256": sha256_file(p), "bytes": p.stat().st_size}
        for p in sorted(stage.rglob("*"))
        if p.is_file()
    }
    (stage / "auditoria" / "MANIFIESTO.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=jdefault), encoding="utf-8"
    )
    for name, spec in schemas.items():
        loaded = pd.read_csv(
            stage / "modelo" / (name + ".csv"), encoding="utf-8-sig", dtype="string"
        )
        if len(loaded) != spec["rows"] or list(loaded.columns) != [
            c["name"] for c in spec["columns"]
        ]:
            raise AssertionError("Exportacion incompleta: " + name)
        if name == "FactMovimientos" and int(pd.to_numeric(loaded.MontoCentavos).sum()) != int(
            tables[name].MontoCentavos.sum()
        ):
            raise AssertionError("Conciliacion monetaria del CSV exportado fallo")
    current = output_root / "modelo_actual"
    old = None
    if current.exists():
        old = output_root / ("_anterior_" + uuid.uuid4().hex[:10])
        if current.resolve().parent != output_root or old.parent != output_root:
            raise ValueError("Movimiento fuera del directorio de salidas")
        current.replace(old)
    try:
        stage.replace(current)
    except Exception:
        if old and (not current.exists()):
            old.replace(current)
        raise
    if old:
        shutil.rmtree(old)
    return current


## Run

Ejecuta el flujo con exclusión de cargas simultáneas y verifica la copia de los seis CSV de consumo.

In [ ]:
def run(archive, output_root=None, config_path=None, iter_path=None):
    output_root = Path(output_root or ROOT / "output").resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    lock = output_root / ".etl.lock"
    fd = os.open(str(lock), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.close(fd)
    try:
        frames, config, source_meta = load_sources(archive)
        if config_path:
            config.update(json.loads(Path(config_path).read_text(encoding="utf-8")))
        local_iter = iter_path or os.environ.get("AH_ITER_ZIP") or config.get("archivo_iter_local")
        if not local_iter:
            official_iter = ROOT / "input" / "iter_00_cpv2020_csv.zip"
            if official_iter.is_file():
                local_iter = official_iter
        print("Fuentes verificadas. Preparando dimensiones y hechos.", flush=True)
        tables, municipios, lineage, tx, metadata = build_model(frames, config, local_iter)
        metadata.update(source_meta)
        qa = validate_model(tables, tx, metadata)
        analyses = summarize(tables)
        current = export_release(
            tables, municipios, lineage, analyses, qa, metadata, output_root.parent / "evidencia"
        )
        expected = {name + ".csv" for name in CSV_TABLES}
        for name in sorted(expected):
            source = current / "modelo" / name
            target = output_root / name
            shutil.copyfile(source, target)
            if sha256_file(source) != sha256_file(target):
                raise AssertionError("Copia incompleta para Power BI: " + name)
        for name in DAX_TABLES:
            obsolete = output_root / (name + ".csv")
            if obsolete.exists():
                if obsolete.resolve().parent != output_root:
                    raise ValueError("CSV obsoleto fuera del directorio de salidas")
                obsolete.unlink()
        result = {
            "tablas_csv": {k: len(tables[k]) for k in CSV_TABLES},
            "dimensiones_dax": list(DAX_TABLES),
            "fecha_corte": metadata["fecha_corte"],
            "validaciones": len(qa),
            "fallidas": 0,
            "enriquecimiento": metadata["enriquecimiento"],
            "resumen": analyses["SensibilidadReglas"].to_dict("records"),
            "directorio": str(output_root),
        }
        print(
            json.dumps(
                {k: v for k, v in result.items() if k != "directorio"},
                ensure_ascii=False,
                indent=2,
                default=jdefault,
            )
        )
        return result
    finally:
        lock.unlink()
